<a href="https://colab.research.google.com/github/solasobambo-prog/ts-data-science/blob/main/week3a1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Your exact initial dataset
customers = pd.DataFrame({
    'customer_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, np.nan],
    'age':     [34, np.nan, 29, 41, np.nan, 52, 38, np.nan, 27, 45, 31, 36],
    'city':    ['Lagos', 'Abuja', None, 'Kano', 'Lagos', None,
                'Ibadan', 'Abuja', 'Lagos', 'Kano', None, 'Lagos'],
    'income':  [250000, np.nan, 180000, np.nan, np.nan, 420000,
                np.nan, 150000, np.nan, 310000, np.nan, 205000],
    'revenue': [1200, 450, 800, 2300, 300, 5100, 640, 275, 980, 3050, 410, 760],
    'churned': ['No', 'Yes', 'No', 'No', 'Yes', 'No',
                'Yes', 'No', 'No', 'No', 'Yes', 'No'],
})

print("--- PART A: DETECT ---")
# 1. Count the missing values in each column
print("Missing Counts:")
print(customers.isna().sum())

# 2. Show the missing percentage for each column, rounded to one decimal place
print("\nMissing Percentages (%):")
print((customers.isna().mean() * 100).round(1))

# 3. Show only the rows where age is missing
print("\nRows where Age is Missing:")
display(customers[customers['age'].isna()])


--- PART A: DETECT ---
Missing Counts:
customer_id    1
age            3
city           3
income         6
revenue        0
churned        0
dtype: int64

Missing Percentages (%):
customer_id     8.3
age            25.0
city           25.0
income         50.0
revenue         0.0
churned         0.0
dtype: float64

Rows where Age is Missing:


,customer_id,age,city,income,revenue,churned
1,102.0,NaN,Abuja,NaN,450,Yes
4,105.0,NaN,Lagos,NaN,300,Yes
7,108.0,NaN,Abuja,150000.0,275,No


In [5]:
print("--- PART B: DROP, FILL, FLAG ---")

# 1. DROP: Remove rows with no customer_id
# Prediction: Row count drops from 12 down to 11 because index 11 has an ID of NaN.
# Comment: customer_id is our unique tracking anchor; observations without it cannot be validated or used.
customers = customers.dropna(subset=['customer_id']).copy()


# 2. FILL: Impute missing age and city profiles
# Expected Age Median: 36.0 (Calculated from remaining valid rows: 34, 29, 41, 52, 38, 27, 45, 31)
# Justification (Age): The median tracks central tendency safely without pulling averages toward extreme age values.
# Justification (City): 'Unknown' explicitly flags missing category entries without losing valid parallel numeric features.
customers['age'] = customers['age'].fillna(customers['age'].median())
customers['city'] = customers['city'].fillna('Unknown')


# 3. FLAG: Flag missing income profiles and evaluate churn correlations
# Prediction: Missing data often signals low-engagement or uncompleted setups, which historically correlate with a positive churn direction.
customers['income_missing'] = customers['income'].isna()

print("\nChurn Rate Comparison by Income Missing Flag (%):")
churn_comparison = (customers['churned'] == 'Yes').groupby(customers['income_missing']).mean() * 100
display(churn_comparison)

# Comment (Income Decision): Keep income with the flag column.
# Dropping the column throws out data for over half the records (54.5%), while dropping rows leaves a sample size too small for meaningful insights.


--- PART B: DROP, FILL, FLAG ---

Churn Rate Comparison by Income Missing Flag (%):


,churned
income_missing,
False,0.000000
True,66.666667


In [2]:
print("--- PART C: CHALLENGE MISSING VALUE REPORT ---")

# Build report using the uncleaned dataset before any dropping or filling occurs
missing_report = pd.DataFrame({
    'missing_count': customers.isna().sum(),
    'missing_pct': (customers.isna().mean() * 100).round(1)
}).sort_values(by='missing_count', ascending=False)

display(missing_report)


--- PART C: CHALLENGE MISSING VALUE REPORT ---


,missing_count,missing_pct
income,6,50.0
age,3,25.0
city,3,25.0
customer_id,1,8.3
revenue,0,0.0
churned,0,0.0
